# Neural Networks: Digit Recognition with an MLP

**Scenario:** Your team is experimenting with a digit recognition system (similar to classifying handwritten forms or invoice numbers). You need a basic neural network model to classify images.

| Assignment step | Where it is covered |
|---|---|
| 2. Prepare data and split | Section 2 |
| 3. Build and train an MLP (TensorFlow/Keras), compare relu / tanh / sigmoid | Section 3 |
| 4. Implement in PyTorch and compare workflows | Section 4 |
| 5. Evaluate (accuracy, confusion matrix) and improve (hyperparameters, layers, optimizers) | Section 5 |

> Requires: `pip install tensorflow torch scikit-learn seaborn matplotlib pandas numpy`

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')
sns.set_theme()

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)

print("TensorFlow:", tf.__version__, "| PyTorch:", torch.__version__)

## 2. Prepare Data and Split

Load the digits dataset (1,797 images of 8x8 pixels, 10 classes). Pixel values run from 0 to 16, so dividing by 16 scales them to [0, 1].

**Splits:** 80% train / 20% test (as specified). I also carve a validation set out of the training data. Hyperparameter and activation choices are made on **validation** accuracy only, so the test set stays untouched until the final evaluation. Splits are stratified so every digit appears in proportion.

In [ ]:
digits = load_digits()
X_digits = digits.data / 16.0          # scale pixels to [0, 1]
y_digits = digits.target

print(f"Samples: {X_digits.shape[0]} | Features: {X_digits.shape[1]} | Classes: {len(np.unique(y_digits))}")
print(f"Pixel range after scaling: {X_digits.min():.2f} to {X_digits.max():.2f}")

# Sample images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(digits.images[i], cmap='gray')
    plt.title(f'Digit: {digits.target[i]}')
    plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=SEED, stratify=y_digits)

# Validation split carved from the training set (used for model selection)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train)

print("Train:     ", X_tr.shape)
print("Validation:", X_val.shape)
print("Test:      ", X_test.shape)

# Class balance
plt.figure(figsize=(8, 3.5))
sns.countplot(x=y_train, color='steelblue')
plt.title('Training set: class distribution')
plt.xlabel('Digit')
plt.tight_layout()
plt.show()

## 3. Build and Train an MLP (TensorFlow/Keras)

The base architecture is a sequential MLP with hidden layers of 128 and 64 units and a 10-way softmax output. `sparse_categorical_crossentropy` is used because labels are integers 0-9.

### 3.1 Baseline model (relu)

In [ ]:
def build_mlp(activation='relu', hidden=(128, 64), optimizer='adam', lr=1e-3, dropout=0.0, seed=SEED):
    """Build and compile a Keras MLP for 64-feature inputs and 10 classes."""
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    m = keras.Sequential([layers.Input(shape=(64,))])
    for units in hidden:
        m.add(layers.Dense(units, activation=activation))
        if dropout > 0:
            m.add(layers.Dropout(dropout))
    m.add(layers.Dense(10, activation='softmax'))

    opts = {
        'adam': keras.optimizers.Adam(lr),
        'sgd': keras.optimizers.SGD(lr, momentum=0.9),
        'rmsprop': keras.optimizers.RMSprop(lr),
    }
    m.compile(optimizer=opts[optimizer],
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    return m

model = build_mlp()
model.summary()

history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=30, batch_size=32, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='validation')
axes[0].set_title('Keras MLP (relu): loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='validation')
axes[1].set_title('Keras MLP (relu): accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Final train accuracy:      {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

### 3.2 Which activation works best? relu vs tanh vs sigmoid

Each activation is trained with 3 random seeds, and validation accuracy is averaged. Using several seeds avoids picking a winner because of one lucky initialization.

In [ ]:
def run_experiment(cfg, seeds=(0, 1, 2), epochs=30):
    """Train cfg with several seeds; return mean/std of final val accuracy and mean val-accuracy curve."""
    finals, curves = [], []
    for s in seeds:
        m = build_mlp(seed=s, **cfg)
        h = m.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=epochs, batch_size=32, verbose=0)
        finals.append(h.history['val_accuracy'][-1])
        curves.append(h.history['val_accuracy'])
    return np.mean(finals), np.std(finals), np.mean(curves, axis=0)

act_results, act_curves = [], {}
for act in ['relu', 'tanh', 'sigmoid']:
    mean_acc, std_acc, curve = run_experiment({'activation': act})
    act_results.append({'Activation': act, 'Mean val accuracy': mean_acc, 'Std': std_acc})
    act_curves[act] = curve

act_df = pd.DataFrame(act_results).sort_values('Mean val accuracy', ascending=False).reset_index(drop=True)
print(act_df.round(4))

plt.figure(figsize=(8, 4.5))
for act, curve in act_curves.items():
    plt.plot(curve, label=act)
plt.title('Validation accuracy by activation (mean of 3 seeds)')
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.legend()
plt.tight_layout()
plt.show()

best_activation = act_df.loc[0, 'Activation']
print("Best activation on validation data:", best_activation)

## 4. Implementation in PyTorch (Comparative)

The same architecture (64 → 128 → 64 → 10, relu) is built in PyTorch with the same optimizer (Adam, lr 1e-3), batch size (32), epochs (30), and data split. Key workflow differences from Keras:

- You define the model as a class with a `forward` method.
- The training loop is written by hand: forward pass, loss, `backward()`, optimizer step.
- `CrossEntropyLoss` expects raw logits, so the model has **no softmax** at the end (softmax is built into the loss).

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def to_loader(X, y, shuffle=False, batch_size=32):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = to_loader(X_tr, y_tr, shuffle=True)
val_loader = to_loader(X_val, y_val)
test_loader = to_loader(X_test, y_test)

class DigitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 10),
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(SEED)
pt_model = DigitMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pt_model.parameters(), lr=1e-3)

def run_epoch(model, loader, train=False):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            out = model(xb)
            loss = criterion(out, yb)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(xb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(xb)
    return total_loss / n, correct / n

pt_hist = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
for epoch in range(30):
    tl, ta = run_epoch(pt_model, train_loader, train=True)
    vl, va = run_epoch(pt_model, val_loader)
    pt_hist['loss'].append(tl); pt_hist['val_loss'].append(vl)
    pt_hist['accuracy'].append(ta); pt_hist['val_accuracy'].append(va)

print(f"Final train accuracy:      {pt_hist['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {pt_hist['val_accuracy'][-1]:.4f}")

# Compare training curves: Keras vs PyTorch
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['val_loss'], label='Keras')
axes[0].plot(pt_hist['val_loss'], label='PyTorch')
axes[0].set_title('Validation loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history.history['val_accuracy'], label='Keras')
axes[1].plot(pt_hist['val_accuracy'], label='PyTorch')
axes[1].set_title('Validation accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Model Evaluation and Improvement

### 5.1 Evaluate the baseline models on the test set

In [ ]:
# Keras baseline
keras_probs = model.predict(X_test, verbose=0)
predictions = keras_probs.argmax(axis=1)
acc_keras = accuracy_score(y_test, predictions)

# PyTorch baseline
pt_model.eval()
with torch.no_grad():
    pt_preds = pt_model(torch.tensor(X_test, dtype=torch.float32).to(device)).argmax(1).cpu().numpy()
acc_pt = accuracy_score(y_test, pt_preds)

print(pd.DataFrame({'Framework': ['Keras', 'PyTorch'],
                    'Test accuracy': [acc_keras, acc_pt]}).round(4))

# Confusion matrix (Keras baseline)
cm = confusion_matrix(y_test, predictions)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Keras baseline: confusion matrix (test set)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.show()

print(classification_report(y_test, predictions))

### 5.2 Experiments: hyperparameters, layers, and optimizers

Each configuration changes one thing from the baseline (relu, hidden layers 128-64, Adam, lr 1e-3, no dropout). All are scored on **validation** accuracy, averaged over 3 seeds.

In [ ]:
experiments = {
    'Baseline (relu, 128-64, adam 1e-3)': {},
    # Layers / width
    'Narrow: 64':                    {'hidden': (64,)},
    'Wide: 256':                     {'hidden': (256,)},
    'Deeper: 128-64-32':             {'hidden': (128, 64, 32)},
    'Larger: 256-128-64':            {'hidden': (256, 128, 64)},
    # Regularization
    'Dropout 0.3':                   {'dropout': 0.3},
    # Optimizers
    'SGD + momentum (lr 0.01)':      {'optimizer': 'sgd', 'lr': 0.01},
    'RMSprop (lr 1e-3)':             {'optimizer': 'rmsprop', 'lr': 1e-3},
    # Learning rates
    'Adam lr 1e-2':                  {'lr': 1e-2},
    'Adam lr 1e-4':                  {'lr': 1e-4},
}

rows, exp_curves = [], {}
for name, cfg in experiments.items():
    mean_acc, std_acc, curve = run_experiment(cfg)
    rows.append({'Experiment': name, 'Mean val accuracy': mean_acc, 'Std': std_acc, 'cfg': cfg})
    exp_curves[name] = curve

exp_df = pd.DataFrame(rows).sort_values('Mean val accuracy', ascending=False).reset_index(drop=True)
print(exp_df[['Experiment', 'Mean val accuracy', 'Std']].round(4))

plt.figure(figsize=(9, 5))
top = exp_df.head(5)['Experiment']
for name in top:
    plt.plot(exp_curves[name], label=name)
plt.title('Top 5 configurations: validation accuracy (mean of 3 seeds)')
plt.xlabel('epoch'); plt.ylabel('validation accuracy'); plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

best_name = exp_df.loc[0, 'Experiment']
best_cfg = exp_df.loc[0, 'cfg']
print("Best configuration on validation data:", best_name, best_cfg)

### 5.3 Final model: best configuration evaluated once on the test set

In [ ]:
final_model = build_mlp(**best_cfg)
final_hist = final_model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                             epochs=30, batch_size=32, verbose=0)

final_preds = final_model.predict(X_test, verbose=0).argmax(axis=1)
acc_final = accuracy_score(y_test, final_preds)

print(f"Best configuration: {best_name}")
print(f"Baseline Keras test accuracy: {acc_keras:.4f}")
print(f"Final model test accuracy:    {acc_final:.4f}")

plt.figure(figsize=(7, 6))
sns.heatmap(confusion_matrix(y_test, final_preds), annot=True, fmt='d', cmap='Greens')
plt.title('Final model: confusion matrix (test set)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.show()

print(classification_report(y_test, final_preds))

# Misclassified examples
wrong = np.where(final_preds != y_test)[0]
print(f"Misclassified: {len(wrong)} of {len(y_test)}")
if len(wrong) > 0:
    n_show = min(10, len(wrong))
    plt.figure(figsize=(12, 3))
    for j, idx in enumerate(wrong[:n_show]):
        plt.subplot(1, n_show, j + 1)
        plt.imshow(X_test[idx].reshape(8, 8), cmap='gray')
        plt.title(f'T:{y_test[idx]} P:{final_preds[idx]}', fontsize=9)
        plt.axis('off')
    plt.suptitle('Misclassified digits (T = true, P = predicted)')
    plt.tight_layout()
    plt.show()

## 6. Findings and Recommendations

*Edit this section after running the notebook and reviewing your own numbers.*

**Activation functions.** Compare the mean validation accuracy in section 3.2. relu and tanh usually train quickly on this dataset, while sigmoid is often slower to converge in 30 epochs. State which activation won and by how much, and whether the gap is larger than the seed-to-seed standard deviation. If it is not, the difference is not meaningful.

**Keras vs PyTorch.** With the same architecture, optimizer, and data, the two frameworks should reach similar accuracy. Small differences come from initialization and batching, not from the framework. Keras is more concise (`compile` and `fit`), while PyTorch exposes the training loop, which gives more control for custom workflows.

**Improvements.** Review the experiment table in section 5.2:
- Did more layers or units help, or did the validation curve show overfitting (train accuracy rising while validation stalls)?
- Which optimizer and learning rate converged best? A learning rate that is too high gives unstable curves, and one that is too low learns slowly in 30 epochs.
- Did dropout help? On a small dataset it often gives a small gain by reducing overfitting.

**Errors.** Use the confusion matrix and the misclassified images to identify which digits get confused (commonly 3/5/8 and 4/9) and whether the images are genuinely ambiguous. Handwritten forms and invoices have far more variation than 8x8 digits.

**Next steps for a real system**
1. Use a convolutional neural network (CNN), which keeps the spatial structure of images and scales to larger inputs.
2. Evaluate with cross-validation or a larger test set, since the test set here is only 360 images.
3. Add early stopping and data augmentation (small shifts and rotations) to improve robustness.
4. Track per-class precision and recall, since some mistakes cost more than others in forms and invoices.